## Using an Individualized Baseline Approach to Predict Reinjury Among MLB Batters 

In [1]:
## Install pybaseball ##

%pip install pybaseball


Note: you may need to restart the kernel to use updated packages.


In [2]:
## Install seaborn ##

%pip install seaborn

## Install openpyxl ##

%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
## Load Relevant Packages ##

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pybaseball as pb

## Read in MLB Injuries Raw Data for 2022 Season ##

url = "https://raw.githubusercontent.com/abrown9008/Sports-Analytics-and-Intelligence-Lab/main/MLB%20Oblique%20Injuries/MLB%20Injuries%20Raw%20Data.xlsx"
injuries22 = pd.read_excel(url, engine="openpyxl",sheet_name='2022')

print(injuries22.head())

              Name                                                URL Team  \
0     Tommy Kahnle  https://www.fangraphs.com/players/tommy-kahnle...  LAD   
1   Caleb Ferguson  https://www.fangraphs.com/players/caleb-fergus...  LAD   
2       Nick Mears  https://www.fangraphs.com/players/nick-mears/2...  PIT   
3  Kyle Funkhouser  https://www.fangraphs.com/players/kyle-funkhou...  DET   
4  Blake Cederlind  https://www.fangraphs.com/players/blake-cederl...  PIT   

  Pos Injury / Surgery Date            Injury / Surgery     Status  \
0  RP   2020-08-04 00:00:00          Tommy John surgery  Activated   
1  RP   2020-09-23 00:00:00          Tommy John surgery  Activated   
2  RP   2021-02-09 00:00:00  Arthroscopic elbow surgery  Activated   
3  RP               Mar '22           Strained shoulder  60-Day IL   
4  RP   2021-03-23 00:00:00          Tommy John surgery  60-Day IL   

  IL Retro Date Eligible to Return Return Date            Latest Update  \
0    2022-04-04         2022-04-14 

In [4]:
## Pivot Data such that each row is a unique player ##

stint_cols = {
    'IL Retro Date': 'IL_Retro',
    'Return Date': 'Activate',
    'Injury / Surgery': 'Injury',
}

df = injuries22.copy()
for c in ['IL Retro Date', 'Return Date']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# Stint 1 = earliest; rows with no IL date sort last
df = df.sort_values(['key_fangraphs', 'IL Retro Date'], na_position='last')
df['stint'] = df.groupby('key_fangraphs').cumcount() + 1

# Player-level attributes (first stint's value is kept if a player changed team)
players = df[['key_fangraphs', 'Name', 'Team', 'Pos', 'URL', 'sub_url']] \
    .drop_duplicates('key_fangraphs')

wide = (
    df.set_index(['key_fangraphs', 'stint'])[list(stint_cols)]
      .rename(columns=stint_cols)
      .unstack('stint')
)
wide.columns = [f'{col}_{n}' for col, n in wide.columns]

max_n = int(df['stint'].max())
ordered = [f'{c}_{n}' for n in range(1, max_n + 1) for c in stint_cols.values()]
wide = wide[ordered].reset_index()

injuries22_wide = players.merge(wide, on='key_fangraphs', how='left')

assert injuries22_wide['key_fangraphs'].is_unique
print(injuries22_wide.shape, df['stint'].value_counts().sort_index().to_dict())



(722, 24) {1: 722, 2: 210, 3: 42, 4: 5, 5: 1, 6: 1}


In [5]:
## Now let's subset to only non-pitchers and oblique injuries for the 2022 season ##

inj22_wide_batters_oblique = injuries22_wide[
    (injuries22_wide['Pos'] != 'P') &
    (injuries22_wide['Pos'] != 'SP') &
    (injuries22_wide['Pos'] != 'RP') &
    (injuries22_wide['Injury_1'].str.contains('oblique', case=False, na=False))
]

In [6]:
## Reusable: read a season sheet and pivot to one row per player ##

stint_cols = {
    'IL Retro Date': 'IL_Retro',
    'Return Date': 'Activate',
    'Injury / Surgery': 'Injury',
}
keep_cols = ['Name', 'URL', 'Team', 'Pos', 'IL Retro Date', 'Return Date',
             'Injury / Surgery', 'sub_url', 'key_fangraphs']

def build_wide(year):
    raw = pd.read_excel(url, engine="openpyxl", sheet_name=str(year))[keep_cols]

    # Recover missing keys from the FanGraphs URL (e.g. /players/name/19931/stats/batting)
    raw['key_fangraphs'] = (
        raw['key_fangraphs']
        .fillna(raw['URL'].str.extract(r'/players/[^/]+/([^/]+)/')[0])
        .astype(str)
    )

    df = raw.copy()
    for c in ['IL Retro Date', 'Return Date']:
        df[c] = pd.to_datetime(df[c], errors='coerce')

    df = df.sort_values(['key_fangraphs', 'IL Retro Date'], na_position='last')
    df['stint'] = df.groupby('key_fangraphs').cumcount() + 1

    players = df[['key_fangraphs', 'Name', 'Team', 'Pos', 'URL', 'sub_url']] \
        .drop_duplicates('key_fangraphs')

    wide = (
        df.set_index(['key_fangraphs', 'stint'])[list(stint_cols)]
          .rename(columns=stint_cols)
          .unstack('stint')
    )
    wide.columns = [f'{col}_{n}' for col, n in wide.columns]
    max_n = int(df['stint'].max())
    ordered = [f'{c}_{n}' for n in range(1, max_n + 1) for c in stint_cols.values()]
    wide = wide[ordered].reset_index()

    out = players.merge(wide, on='key_fangraphs', how='left')
    assert out['key_fangraphs'].is_unique
    assert out['key_fangraphs'].ne('nan').all(), f"{year}: unresolved keys"
    return out

## Build one wide dataframe per season ##

injuries_wide = {yr: build_wide(yr) for yr in [2021, 2022, 2023, 2024, 2025, 2026]}

injuries21_wide = injuries_wide[2021]
injuries23_wide = injuries_wide[2023]
injuries24_wide = injuries_wide[2024]
injuries25_wide = injuries_wide[2025]
injuries26_wide = injuries_wide[2026]

for yr, d in injuries_wide.items():
    print(yr, d.shape)

2021 (843, 18)
2022 (722, 24)
2023 (677, 18)
2024 (619, 18)
2025 (634, 18)
2026 (675, 18)


In [7]:
## Row bind and add season column ##

all_wide = pd.concat(
    [d.assign(Season=yr) for yr, d in injuries_wide.items()],
    ignore_index=True
)

In [8]:
## Subset all_wide to only non-pitchers and oblique injuries for all seasons ##

all_wide_batters_oblique = all_wide[
    (all_wide['Pos'] != 'P') &
    (all_wide['Pos'] != 'SP') &
    (all_wide['Pos'] != 'RP') &
    (all_wide['Injury_1'].str.contains('oblique', case=False, na=False))
]

In [9]:
## Remove players with NA in Active_1 column ##

all_wide_batters_oblique = all_wide_batters_oblique[
    all_wide_batters_oblique['Activate_1'].notna()
]

In [10]:
## How many players whose first injury was an oblique injury experienced a second oblique injury? ##

print("Total Number of Players with Oblique Injuries:", all_wide_batters_oblique.shape[0])
print("Number of Players with a Second Injury:", all_wide_batters_oblique['Injury_2'].notna().sum())
print("Number of Players with a Second Oblique Injury:", all_wide_batters_oblique['Injury_2'].str.contains('oblique', case=False, na=False).sum())

Total Number of Players with Oblique Injuries: 157
Number of Players with a Second Injury: 43
Number of Players with a Second Oblique Injury: 7


In [11]:
## Print the types of second injuries for players whose first injury was an oblique injury ##

print("Types of Second Injuries for Players with First Oblique Injury:")
print(all_wide_batters_oblique['Injury_2'].value_counts(dropna=False))

Types of Second Injuries for Players with First Oblique Injury:
Injury_2
NaN                                       114
Strained hamstring                          8
Strained oblique                            7
Strained quad                               3
Sprained ankle                              2
Knee inflammation                           1
Sprained thumb                              1
Shoulder subluxation (left)                 1
Fractured finger (left pinky)               1
Knee discomfort (patellar subluxation)      1
Sprained knee                               1
Forearm surgery (fracture)                  1
Lower back inflammation                     1
Plantar fascitis                            1
Fractured hand (right pinkie)               1
Strained lower back                         1
Back inflammation                           1
Fractured thumb (left)                      1
Ankle/foot tendinitis                       1
Back spasms                                 1
Spraine

In [12]:
## Okay so we're going to group secondary injuries into categories. Let's define a function to do that. ##

# ---------------------------------------------------------------------------
# 1. Exact-match mapping for the injury labels currently in the data.
#    Editing this dict is the place to record judgment calls (e.g., hamate).
# ---------------------------------------------------------------------------
EXACT_MAP = {
    "Strained oblique": "oblique_recurrence",
 
    "Strained hamstring": "lower_body_strain",
    "Strained quad": "lower_body_strain",
    "Strained calf": "lower_body_strain",
    "Strained hip flexor": "lower_body_strain",
 
    "Strained lower back": "lower_back",
    "Lower back inflammation": "lower_back",
    "Back inflammation": "lower_back",
    "Back spasms": "lower_back",
 
    "Knee inflammation": "knee_nontraumatic",
    "Knee discomfort (patellar subluxation)": "knee_nontraumatic",
    "Sprained knee": "knee_nontraumatic",
    "Arthroscopic knee surgery": "knee_nontraumatic",
 
    "Plantar fascitis": "other_overuse",
    "Ankle/foot tendinitis": "other_overuse",
    "Hand inflammation": "other_overuse",
 
    "Sprained ankle": "ankle_sprain",
 
    "Fractured finger (left pinky)": "traumatic",
    "Fractured hand (right pinkie)": "traumatic",
    "Fractured thumb (left)": "traumatic",
    "Forearm surgery (fracture)": "traumatic",
    "Hamate surgery (left hand)": "traumatic",  # judgment call: possibly swing-related
    "Sprained thumb": "traumatic",
    "Sprained hand (left)": "traumatic",
    "Fractured knee": "traumatic",
    "Knee bone bruise": "traumatic",
    "Shoulder subluxation (left)": "traumatic",
}
 
# ---------------------------------------------------------------------------
# 2. Keyword fallback for labels not in EXACT_MAP (e.g., when you add seasons).
#    Order matters: the first matching rule wins. Anything matched here is
#    flagged so you can review it and add it to EXACT_MAP.
# ---------------------------------------------------------------------------
KEYWORD_RULES = [
    (["fracture", "hamate", "bruise", "contusion", "hit by pitch", "concussion",
      "subluxation (left)", "subluxation (right)", "dislocat"], "traumatic"),
    (["oblique"], "oblique_recurrence"),
    (["back", "lumbar", "spasm"], "lower_back"),
    (["hamstring", "quad", "calf", "hip flexor", "groin", "adductor"], "lower_body_strain"),
    (["knee", "patella", "meniscus"], "knee_nontraumatic"),
    (["ankle sprain", "sprained ankle"], "ankle_sprain"),
    (["sprained hand", "sprained thumb", "sprained finger", "sprained wrist"], "traumatic"),
    (["tendinitis", "tendinopathy", "fasci", "inflammation", "bursitis"], "other_overuse"),
]
 
PRIMARY_GROUPS = {"oblique_recurrence", "lower_body_strain", "lower_back"}
SENSITIVITY_GROUPS = PRIMARY_GROUPS | {"knee_nontraumatic", "other_overuse", "ankle_sprain"}
NEGATIVE_CONTROL_GROUPS = {"traumatic"}
 
 
def classify_injury(label):
    """Return (group, match_method) for a single injury label."""
    if pd.isna(label) or str(label).strip() == "":
        return "none", "none"
 
    label = str(label).strip()
    if label in EXACT_MAP:
        return EXACT_MAP[label], "exact"
 
    lowered = label.lower()
    for keywords, group in KEYWORD_RULES:
        if any(k in lowered for k in keywords):
            return group, "keyword"
 
    return "unclassified", "unmatched"
 
 
def add_injury_groups(df, injury_col="Injury_2"):
    """Add group, outcome flags, and competing-risks event codes to df."""
    out = df.copy()
    results = out[injury_col].apply(classify_injury)
    out["injury_group"] = results.str[0]
    out["match_method"] = results.str[1]
 
    has_injury = out["injury_group"] != "none"
    out["primary_outcome"] = out["injury_group"].isin(PRIMARY_GROUPS).astype(int)
    out["sensitivity_outcome"] = out["injury_group"].isin(SENSITIVITY_GROUPS).astype(int)
    out["negative_control"] = out["injury_group"].isin(NEGATIVE_CONTROL_GROUPS).astype(int)
 
    # Competing-risks event codes: 0 = censored, 1 = event of interest, 2 = competing
    out["event_primary"] = np.select(
        [out["primary_outcome"] == 1, has_injury],
        [1, 2],
        default=0,
    )
    out["event_sensitivity"] = np.select(
        [out["sensitivity_outcome"] == 1, has_injury],
        [1, 2],
        default=0,
    )
 
    # Warn about anything that wasn't an exact match
    review = out.loc[out["match_method"].isin(["keyword", "unmatched"]), injury_col]
    if not review.empty:
        print("WARNING: these labels were not exact matches; review and add to EXACT_MAP:")
        for label, method in (
            out.loc[review.index, [injury_col, "match_method"]]
            .drop_duplicates()
            .itertuples(index=False)
        ):
            print(f"  [{method}] {label!r} -> {classify_injury(label)[0]}")
        print()
 
    return out
 
 
def summarize(grouped):
    """Print group counts and outcome totals."""
    print("Counts by injury group:")
    print(grouped["injury_group"].value_counts().to_string())
    print()
    injured = grouped["injury_group"] != "none"
    print(f"Players with a second injury: {injured.sum()} of {len(grouped)}")
    print(f"Primary outcome events:      {grouped['primary_outcome'].sum()}")
    print(f"Sensitivity outcome events:  {grouped['sensitivity_outcome'].sum()}")
    print(f"Negative control events:     {grouped['negative_control'].sum()}")
 
 
if __name__ == "__main__":
    # Example using the counts from the current data. Replace this block with
    # your own DataFrame, e.g.: df = pd.read_csv("oblique_players.csv")
    counts = {
        np.nan: 114,
        "Strained hamstring": 8, "Strained oblique": 7, "Strained quad": 3,
        "Sprained ankle": 2, "Knee inflammation": 1, "Sprained thumb": 1,
        "Shoulder subluxation (left)": 1, "Fractured finger (left pinky)": 1,
        "Knee discomfort (patellar subluxation)": 1, "Sprained knee": 1,
        "Forearm surgery (fracture)": 1, "Lower back inflammation": 1,
        "Plantar fascitis": 1, "Fractured hand (right pinkie)": 1,
        "Strained lower back": 1, "Back inflammation": 1,
        "Fractured thumb (left)": 1, "Ankle/foot tendinitis": 1,
        "Back spasms": 1, "Sprained hand (left)": 1, "Strained calf": 1,
        "Hand inflammation": 1, "Hamate surgery (left hand)": 1,
        "Arthroscopic knee surgery": 1, "Fractured knee": 1,
        "Strained hip flexor": 1, "Knee bone bruise": 1,
    }
    df = pd.DataFrame({"Injury_2": [k for k, n in counts.items() for _ in range(n)]})
 
    grouped = add_injury_groups(df, injury_col="Injury_2")
    summarize(grouped)

Counts by injury group:
injury_group
none                  114
lower_body_strain      13
traumatic              10
oblique_recurrence      7
knee_nontraumatic       4
lower_back              4
other_overuse           3
ankle_sprain            2

Players with a second injury: 43 of 157
Primary outcome events:      24
Sensitivity outcome events:  33
Negative control events:     10


In [13]:
grouped = add_injury_groups(all_wide_batters_oblique, injury_col="Injury_2")
summarize(grouped)

Counts by injury group:
injury_group
none                  114
lower_body_strain      13
traumatic              10
oblique_recurrence      7
knee_nontraumatic       4
lower_back              4
other_overuse           3
ankle_sprain            2

Players with a second injury: 43 of 157
Primary outcome events:      24
Sensitivity outcome events:  33
Negative control events:     10


In [14]:
"""
Game-by-game wOBA from Statcast via pybaseball.

Statcast assigns each plate-appearance-ending pitch a woba_value (the run value
weight for that outcome) and a woba_denom (1 if the PA counts toward wOBA, else 0).
Game wOBA = sum(woba_value) / sum(woba_denom).

Requires:  pip install pybaseball pandas numpy
"""

import time
from pybaseball import cache, playerid_lookup, playerid_reverse_lookup, statcast_batter

cache.enable()


def lookup_mlbam_id(last, first):
    """Return the MLBAM ID for a player (most recent if several share the name)."""
    result = playerid_lookup(last, first)
    if result.empty:
        return None
    result = result.sort_values("mlb_played_last", ascending=False)
    return int(result.iloc[0]["key_mlbam"])


def add_mlbam_ids(df, fg_col="key_fangraphs", out_col="mlbam_id"):
    """
    Add an MLBAM ID column by mapping FanGraphs IDs through the Chadwick register.
    Prints any players who couldn't be matched so you can fill them in by hand.
    """
    out = df.copy()
    # FanGraphs IDs are numeric for MLB players; prospect IDs (e.g., "sa123456") won't match
    fg_ids = pd.to_numeric(out[fg_col], errors="coerce").dropna().astype(int).unique().tolist()

    lookup = playerid_reverse_lookup(fg_ids, key_type="fangraphs")
    id_map = dict(zip(lookup["key_fangraphs"].astype(int), lookup["key_mlbam"].astype(int)))

    out[out_col] = pd.to_numeric(out[fg_col], errors="coerce").map(id_map).astype("Int64")

    missing = out.loc[out[out_col].isna(), fg_col].unique()
    if len(missing):
        print(f"{len(missing)} FanGraphs IDs had no MLBAM match: {list(missing)}")
    return out


def get_game_woba(mlbam_id, season, game_types=("R",)):
    """
    One row per game: wOBA numerator, denominator, and game wOBA.
    Also returns xwOBA (expected wOBA from contact quality) as a less noisy companion.
    """
    df = statcast_batter(f"{season}-03-01", f"{season}-11-30", player_id=mlbam_id)
    if df is None or df.empty:
        return pd.DataFrame()

    df = df[df["game_type"].isin(game_types)]
    df = df[df["woba_denom"].notna()].copy()  # keep only PA-ending pitches

    # xwOBA: use the contact-based estimate on balls in play, actual woba_value otherwise
    est = df["estimated_woba_using_speedangle"]
    df["xwoba_value"] = np.where(est.notna(), est, df["woba_value"])

    log = (
        df.groupby(["game_pk", "game_date"])
        .agg(woba_num=("woba_value", "sum"),
             woba_denom=("woba_denom", "sum"),
             xwoba_num=("xwoba_value", "sum"))
        .reset_index()
    )
    log = log[log["woba_denom"] > 0]
    log["woba"] = log["woba_num"] / log["woba_denom"]
    log["xwoba"] = log["xwoba_num"] / log["woba_denom"]

    log["game_date"] = pd.to_datetime(log["game_date"])
    log = log.sort_values("game_date").reset_index(drop=True)
    log.insert(0, "mlbam_id", mlbam_id)
    log.insert(1, "season", season)
    log["game_num"] = np.arange(1, len(log) + 1)
    return log


def get_cohort_game_woba(cohort, id_col="mlbam_id", season_col="season", pause=2.0):
    """Stack game-by-game wOBA for every player-season in a cohort DataFrame."""
    cols = [id_col, season_col]
    logs, failures = [], []
    for row in cohort[cols].drop_duplicates().itertuples(index=False):
        mlbam_id, season = int(row[0]), int(row[1])
        try:
            log = get_game_woba(mlbam_id, season)
            (logs if not log.empty else failures).append(
                log if not log.empty else (mlbam_id, season, "no data"))
        except Exception as e:
            failures.append((mlbam_id, season, str(e)))
        time.sleep(pause)

    if failures:
        print(f"{len(failures)} player-seasons failed:", *failures, sep="\n  ")
    return pd.concat(logs, ignore_index=True) if logs else pd.DataFrame()


if __name__ == "__main__":
    pid = lookup_mlbam_id("judge", "aaron")
    print(get_game_woba(pid, 2024).head(10).to_string())

   mlbam_id  season  game_pk  game_date  woba_num  woba_denom  xwoba_num   woba     xwoba  game_num
0    592450    2024   746418 2024-03-28      1.95         5.0   1.520400  0.390  0.304080         1
1    592450    2024   746412 2024-03-29      0.00         5.0   0.391000  0.000  0.078200         2
2    592450    2024   746410 2024-03-30      1.60         4.0   1.609131  0.400  0.402283         3
3    592450    2024   746413 2024-03-31      0.00         5.0   0.979000  0.000  0.195800         4
4    592450    2024   747218 2024-04-01      0.90         5.0   1.982000  0.180  0.396400         5
5    592450    2024   747220 2024-04-02      0.70         4.0   1.039131  0.175  0.259783         6
6    592450    2024   747219 2024-04-03      3.95         5.0   3.138909  0.790  0.627782         7
7    592450    2024   745764 2024-04-05      1.40         4.0   1.453262  0.350  0.363316         8
8    592450    2024   745766 2024-04-06      4.30         5.0   3.860262  0.860  0.772052         9


In [15]:
grouped = add_mlbam_ids(grouped, fg_col="key_fangraphs")
cohort_with_ids = grouped.dropna(subset=["mlbam_id", "Season"]).copy()
cohort_with_ids["mlbam_id"] = cohort_with_ids["mlbam_id"].astype(int)
cohort_with_ids["Season"] = cohort_with_ids["Season"].astype(int)

woba_logs = get_cohort_game_woba(
    cohort_with_ids,
    id_col="mlbam_id",
    season_col="Season",
)

3 FanGraphs IDs had no MLBAM match: ['26468', '33270', '37122']


Gathering Player Data
Gathering Player Data
3 player-seasons failed:
  (592767, 2022, 'no data')
  (688760, 2026, 'no data')
  (691185, 2026, 'no data')


In [16]:
## Define league average wOBA and xwOBA across all seasons by creating
## a wOBA_ref pandas DataFrame manually (cite Fangraphs) ##

wOBA_ref = pd.DataFrame({
    "season": [2021, 2022, 2023, 2024, 2025, 2026],
    "woba": [0.314,0.310,0.318,0.310,0.313,0.316],
    "xwoba": [0.317,0.312,0.320,0.316,0.316,0.314]
})

In [17]:
## For each player, I now need to calculate their reference value,
## which is defined as:
## wOBA_ref = if(num_games_prior_to_injury == 0), then league average wOBA for that season,
## else, wOBA_ref = (num_games_prior_to_injury * player_mean_wOBA_prior_to_injury + 7*league_average_wOBA_for_that_season) / 
## (num_games_prior_to_injury + 7) ##

PRIOR_WEIGHT = 7

## Games played before the first (oblique) injury, per player-season ##
logs = woba_logs.merge(
    grouped[['mlbam_id', 'Season', 'IL_Retro_1']].dropna(subset=['mlbam_id', 'IL_Retro_1']),
    left_on=['mlbam_id', 'season'],
    right_on=['mlbam_id', 'Season'],
)
prior = (
    logs[logs['game_date'] < logs['IL_Retro_1']]
    .groupby(['mlbam_id', 'season'])['woba']
    .agg(num_games_prior_to_injury='count', player_mean_wOBA_prior_to_injury='mean')
    .reset_index()
    .rename(columns={'season': 'Season'})
)

grouped = grouped.drop(
    columns=['num_games_prior_to_injury', 'player_mean_wOBA_prior_to_injury',
             'league_avg_wOBA', 'wOBA_ref'],
    errors='ignore',
)
grouped = (
    grouped
    .merge(prior, on=['mlbam_id', 'Season'], how='left')
    .merge(wOBA_ref[['season', 'woba']].rename(columns={'season': 'Season', 'woba': 'league_avg_wOBA'}),
           on='Season', how='left')
)
grouped['num_games_prior_to_injury'] = grouped['num_games_prior_to_injury'].fillna(0).astype(int)

n = grouped['num_games_prior_to_injury']
grouped['wOBA_ref'] = np.where(
    n == 0,
    grouped['league_avg_wOBA'],
    (n * grouped['player_mean_wOBA_prior_to_injury'].fillna(0)
     + PRIOR_WEIGHT * grouped['league_avg_wOBA']) / (n + PRIOR_WEIGHT),
)

print(grouped[['Name', 'Season', 'num_games_prior_to_injury',
               'player_mean_wOBA_prior_to_injury', 'league_avg_wOBA', 'wOBA_ref']].head(10))

                Name  Season  num_games_prior_to_injury  \
0       Brandon Belt    2021                         40   
1        Kolten Wong    2021                          7   
2    George Springer    2021                          0   
3   Francisco Lindor    2021                         88   
4          Max Muncy    2021                         59   
5      Erik González    2021                         71   
6   Christian Walker    2021                         10   
7  Adalberto Mondesi    2021                          0   
8          Tim Lopes    2021                          0   
9       Nomar Mazara    2021                         12   

   player_mean_wOBA_prior_to_injury  league_avg_wOBA  wOBA_ref  
0                          0.334687            0.314  0.331606  
1                          0.222857            0.314  0.268429  
2                               NaN            0.314  0.314000  
3                          0.312188            0.314  0.312321  
4                        

In [ ]:
"""
Game-by-game wOBA from Statcast via pybaseball.

Statcast assigns each plate-appearance-ending pitch a woba_value (the run value
weight for that outcome) and a woba_denom (1 if the PA counts toward wOBA, else 0).
Game wOBA = sum(woba_value) / sum(woba_denom).

Requires:  pip install pybaseball pandas numpy
"""

import time

import numpy as np
import pandas as pd
from pybaseball import cache, playerid_lookup, playerid_reverse_lookup, statcast, statcast_batter

cache.enable()


def lookup_mlbam_id(last, first):
    """Return the MLBAM ID for a player (most recent if several share the name)."""
    result = playerid_lookup(last, first)
    if result.empty:
        return None
    result = result.sort_values("mlb_played_last", ascending=False)
    return int(result.iloc[0]["key_mlbam"])


def add_mlbam_ids(df, fg_col="key_fangraphs", out_col="mlbam_id"):
    """
    Add an MLBAM ID column by mapping FanGraphs IDs through the Chadwick register.
    Prints any players who couldn't be matched so you can fill them in by hand.
    """
    out = df.copy()
    # FanGraphs IDs are numeric for MLB players; prospect IDs (e.g., "sa123456") won't match
    fg_ids = pd.to_numeric(out[fg_col], errors="coerce").dropna().astype(int).unique().tolist()

    lookup = playerid_reverse_lookup(fg_ids, key_type="fangraphs")
    id_map = dict(zip(lookup["key_fangraphs"].astype(int), lookup["key_mlbam"].astype(int)))

    out[out_col] = pd.to_numeric(out[fg_col], errors="coerce").map(id_map).astype("Int64")

    missing = out.loc[out[out_col].isna(), fg_col].unique()
    if len(missing):
        print(f"{len(missing)} FanGraphs IDs had no MLBAM match: {list(missing)}")
    return out


def get_game_woba(mlbam_id, season, game_types=("R",)):
    """
    One row per game: wOBA numerator, denominator, and game wOBA.
    """
    df = statcast_batter(f"{season}-03-01", f"{season}-11-30", player_id=mlbam_id)
    if df is None or df.empty:
        return pd.DataFrame()

    df = df[df["game_type"].isin(game_types)]
    df = df[df["woba_denom"].notna()].copy()  # keep only PA-ending pitches

    log = (
        df.groupby(["game_pk", "game_date"])
        .agg(woba_num=("woba_value", "sum"),
             woba_denom=("woba_denom", "sum"))
        .reset_index()
    )
    log = log[log["woba_denom"] > 0]
    log["woba"] = log["woba_num"] / log["woba_denom"]

    log["game_date"] = pd.to_datetime(log["game_date"])
    log = log.sort_values("game_date").reset_index(drop=True)
    log.insert(0, "mlbam_id", mlbam_id)
    log.insert(1, "season", season)
    log["game_num"] = np.arange(1, len(log) + 1)
    return log


def get_cohort_game_woba(cohort, id_col="mlbam_id", season_col="season", pause=2.0):
    """Stack game-by-game wOBA for every player-season in a cohort DataFrame."""
    cols = [id_col, season_col]
    logs, failures = [], []
    for row in cohort[cols].drop_duplicates().itertuples(index=False):
        mlbam_id, season = int(row[0]), int(row[1])
        try:
            log = get_game_woba(mlbam_id, season)
            (logs if not log.empty else failures).append(
                log if not log.empty else (mlbam_id, season, "no data"))
        except Exception as e:
            failures.append((mlbam_id, season, str(e)))
        time.sleep(pause)

    if failures:
        print(f"{len(failures)} player-seasons failed:", *failures, sep="\n  ")
    return pd.concat(logs, ignore_index=True) if logs else pd.DataFrame()


# ---------------------------------------------------------------------------
# League-level within-player variation in game wOBA
# ---------------------------------------------------------------------------
def get_league_game_woba(season, game_types=("R",), max_pitcher_pa=100):
    """
    Game-by-game wOBA for EVERY batter in a season (one Statcast pull for the season).

    Pitchers hitting (common before the 2022 universal DH) are excluded: any batter
    who also threw 100+ pitches that season and had fewer than max_pitcher_pa PA.
    Two-way players with real PA totals (e.g., Ohtani) are kept.
    Note: a full-season pull is large and slow (tens of minutes); the cache helps.
    """
    df = statcast(f"{season}-03-01", f"{season}-11-30")
    df = df[df["game_type"].isin(game_types)]

    pitch_counts = df.groupby("pitcher").size()
    pitchers = set(pitch_counts[pitch_counts >= 100].index)

    pa = df[df["woba_denom"].notna()].copy()
    pa_per_batter = pa.groupby("batter")["woba_denom"].sum()
    drop = {b for b in pitchers if pa_per_batter.get(b, 0) < max_pitcher_pa}
    pa = pa[~pa["batter"].isin(drop)]

    log = (
        pa.groupby(["batter", "game_pk", "game_date"])
        .agg(woba_num=("woba_value", "sum"), woba_denom=("woba_denom", "sum"))
        .reset_index()
        .rename(columns={"batter": "mlbam_id"})
    )
    log = log[log["woba_denom"] > 0]
    log["woba"] = log["woba_num"] / log["woba_denom"]
    log["season"] = season

    # Keep the PA-level values too, for the per-PA variance estimate
    pa_level = pa[["batter", "woba_value", "woba_denom"]].rename(columns={"batter": "mlbam_id"})
    pa_level = pa_level[pa_level["woba_denom"] > 0]
    pa_level["season"] = season
    return log, pa_level


def league_within_player_variation(game_logs, pa_level=None, min_games=20):
    """
    Per season, summarize how much a typical player's game wOBA varies around
    his own mean.

    Returns per season:
      league_avg_wOBA      PA-weighted league wOBA (consistent with Statcast values)
      n_players            players with at least min_games games
      mean_within_sd       simple mean of each player's game-wOBA SD
      pooled_within_sd     sqrt( sum((n_i - 1) * s_i^2) / sum(n_i - 1) )
      mean_pa_per_game     average PA per game among included players
      pooled_within_sd_pa  pooled within-player SD of single-PA wOBA values
                           (game-level SD for a game with k PA ~ this / sqrt(k))
    """
    rows = []
    for season, g in game_logs.groupby("season"):
        league_avg = g["woba_num"].sum() / g["woba_denom"].sum()

        per_player = g.groupby("mlbam_id")["woba"].agg(n="count", var="var")
        per_player = per_player[per_player["n"] >= min_games]
        keep = set(per_player.index)

        pooled_var = ((per_player["n"] - 1) * per_player["var"]).sum() / (per_player["n"] - 1).sum()

        row = {
            "season": season,
            "league_avg_wOBA": league_avg,
            "n_players": len(per_player),
            "mean_within_sd": np.sqrt(per_player["var"]).mean(),
            "pooled_within_sd": np.sqrt(pooled_var),
            "mean_pa_per_game": g.loc[g["mlbam_id"].isin(keep), "woba_denom"].mean(),
        }

        if pa_level is not None:
            p = pa_level[(pa_level["season"] == season) & pa_level["mlbam_id"].isin(keep)]
            pp = p.groupby("mlbam_id")["woba_value"].agg(n="count", var="var")
            row["pooled_within_sd_pa"] = np.sqrt(
                ((pp["n"] - 1) * pp["var"]).sum() / (pp["n"] - 1).sum()
            )
        rows.append(row)

    return pd.DataFrame(rows)


def get_league_variation(seasons, min_games=20):
    """Pull every season and return the league-level variation table."""
    logs, pas = [], []
    for season in seasons:
        log, pa_level = get_league_game_woba(season)
        logs.append(log)
        pas.append(pa_level)
    return league_within_player_variation(
        pd.concat(logs, ignore_index=True), pd.concat(pas, ignore_index=True), min_games
    )


if __name__ == "__main__":
    pid = lookup_mlbam_id("judge", "aaron")
    print(get_game_woba(pid, 2024).head(10).to_string())

   mlbam_id  season  game_pk  game_date  woba_num  woba_denom   woba  game_num
0    592450    2024   746418 2024-03-28      1.95         5.0  0.390         1
1    592450    2024   746412 2024-03-29      0.00         5.0  0.000         2
2    592450    2024   746410 2024-03-30      1.60         4.0  0.400         3
3    592450    2024   746413 2024-03-31      0.00         5.0  0.000         4
4    592450    2024   747218 2024-04-01      0.90         5.0  0.180         5
5    592450    2024   747220 2024-04-02      0.70         4.0  0.175         6
6    592450    2024   747219 2024-04-03      3.95         5.0  0.790         7
7    592450    2024   745764 2024-04-05      1.40         4.0  0.350         8
8    592450    2024   745766 2024-04-06      4.30         5.0  0.860         9
9    592450    2024   745765 2024-04-07      1.40         5.0  0.280        10


: 

In [ ]:
"""
Run the full wOBA pipeline on the in-memory `grouped` dataframe.

grouped needs key_fangraphs and Season columns.
Outputs are written to ./pipeline_output/ and reused on reruns, so if a pull
fails partway through, just run the script again and it picks up where it left off.
"""

import sys
from pathlib import Path

SEASONS = range(2021, 2027)
MIN_GAMES = 20
OUT = Path("pipeline_output")
OUT.mkdir(exist_ok=True)


def main(grouped_df):
    grouped = grouped_df

    # 1. FanGraphs -> MLBAM IDs
    grouped = add_mlbam_ids(grouped, fg_col="key_fangraphs")
    grouped.to_csv(OUT / "grouped_with_ids.csv", index=False)

    # 2. Cohort game-by-game wOBA
    cohort_path = OUT / "cohort_woba_logs.csv"
    if cohort_path.exists():
        woba_logs = pd.read_csv(cohort_path, parse_dates=["game_date"])
        print(f"Loaded cohort logs from {cohort_path}")
    else:
        cohort = grouped.dropna(subset=["mlbam_id"])
        woba_logs = get_cohort_game_woba(cohort, id_col="mlbam_id", season_col="Season")
        woba_logs.to_csv(cohort_path, index=False)
        print(f"Saved {len(woba_logs)} player-games to {cohort_path}")

    # 3. League game logs, one season at a time (slow: full-season Statcast pulls)
    league_logs, league_pas = [], []
    for season in SEASONS:
        log_path = OUT / f"league_game_woba_{season}.csv"
        pa_path = OUT / f"league_pa_woba_{season}.csv"
        if log_path.exists() and pa_path.exists():
            log, pa = pd.read_csv(log_path), pd.read_csv(pa_path)
            print(f"{season}: loaded from disk")
        else:
            print(f"{season}: pulling full season from Statcast (this takes a while)...")
            log, pa = get_league_game_woba(season)
            log.to_csv(log_path, index=False)
            pa.to_csv(pa_path, index=False)
        league_logs.append(log)
        league_pas.append(pa)

    # 4. League-level within-player variation
    league_var = league_within_player_variation(
        pd.concat(league_logs, ignore_index=True),
        pd.concat(league_pas, ignore_index=True),
        min_games=MIN_GAMES,
    )
    league_var.to_csv(OUT / "league_variation.csv", index=False)
    print("\nLeague-level within-player variation:")
    print(league_var.to_string(index=False))


main(grouped)

3 FanGraphs IDs had no MLBAM match: ['26468', '33270', '37122']
Loaded cohort logs from pipeline_output/cohort_woba_logs.csv
2021: pulling full season from Statcast (this takes a while)...
This is a large query, it may take a moment to complete


Skipping offseason dates
Skipping offseason dates


100%|██████████| 246/246 [00:24<00:00, 10.18it/s]
